# HP filter: synthetic macro-style recovery (no MCP)

Ground truth $y_t = g_t + c_t + \varepsilon_t$ with explicit smooth trend $g_t$, oscillatory cycle $c_t$, and Gaussian noise. Standard Hodrick–Prescott cycle $\hat{c}(\lambda) = y - \hat{g}(\lambda)$ for a sparse $\log_{10}$ grid of $\lambda \in [10^2, 10^9]$.

**Oracle metric:** $\mathrm{RMSE}(\hat{c} - c^{\mathrm{true}}) / \mathrm{sd}(c^{\mathrm{true}})$ vs $\log_{10}\lambda$; overlays at grid-best and grid-worst $\lambda$.


## Setup

NumPy + Matplotlib only. HP trend solves $(I + \lambda D_2^\top D_2)\,\hat{g} = y$ with second-difference matrix $D_2$ (rows $t=1,\ldots,T-2$ in 0-based indexing: $\hat{g}_t - 2\hat{g}_{t-1} + \hat{g}_{t-2}$).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(20260503)

T = 240  # ≥ 200
t = np.arange(T, dtype=float)

# Explicit components
g = 0.018 * t + 8e-6 * (t - T / 2) ** 2
c = (
    1.35 * np.sin(2 * np.pi * t / 42.0)
    + 0.55 * np.sin(2 * np.pi * t / 17.0)
    + 0.25 * np.sin(2 * np.pi * t / 11.0)
)
sigma_eps = 0.28
eps = rng.normal(loc=0.0, scale=sigma_eps, size=T)
y = g + c + eps

print("T", T)
print("y.shape", y.shape, "g.shape", g.shape, "c.shape", c.shape, "eps.shape", eps.shape)
print("std(c)", float(np.std(c, ddof=0)))


In [ ]:
def hp_trend_cycle(y: np.ndarray, lam: float) -> tuple[np.ndarray, np.ndarray]:
    """HP decomposition: minimize ||y-g||^2 + lam ||D2 g||^2. Returns (g_hat, c_hat)."""
    y = np.asarray(y, dtype=float).ravel()
    Tn = y.size
    if Tn < 3:
        raise ValueError("need T>=3")
    D2 = np.zeros((Tn - 2, Tn))
    for i in range(Tn - 2):
        D2[i, i] = 1.0
        D2[i, i + 1] = -2.0
        D2[i, i + 2] = 1.0
    A = np.eye(Tn) + lam * (D2.T @ D2)
    g_hat = np.linalg.solve(A, y)
    c_hat = y - g_hat
    return g_hat, c_hat


def oracle_rmse_norm(c_hat: np.ndarray, c_true: np.ndarray) -> float:
    err = c_hat - c_true
    rmse = float(np.sqrt(np.mean(err * err)))
    denom = float(np.std(c_true, ddof=0))
    return rmse / denom


In [ ]:
# Sparse log-spaced grid: 10 points in [1e2, 1e9] (≥8)
lams = np.logspace(2, 9, num=10)
log10_lam = np.log10(lams)

rmse_norm = np.empty_like(lams)
cycles = []
for lam in lams:
    _, chat = hp_trend_cycle(y, float(lam))
    cycles.append(chat)
    rmse_norm[len(cycles) - 1] = oracle_rmse_norm(chat, c)

cycles = np.stack(cycles, axis=0)
i_best = int(np.argmin(rmse_norm))
i_worst = int(np.argmax(rmse_norm))
lam_best, lam_worst = float(lams[i_best]), float(lams[i_worst])

print("λ grid count", len(lams))
print("best  λ", lam_best, "log10", float(log10_lam[i_best]), "oracle RMSE/sd(c)", float(rmse_norm[i_best]))
print("worst λ", lam_worst, "log10", float(log10_lam[i_worst]), "oracle RMSE/sd(c)", float(rmse_norm[i_worst]))


In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 4.0))
ax.plot(log10_lam, rmse_norm, "o-", color="0.2", lw=1.2, ms=6)
ax.scatter([log10_lam[i_best]], [rmse_norm[i_best]], color="C2", s=90, zorder=3, label="grid-best")
ax.scatter([log10_lam[i_worst]], [rmse_norm[i_worst]], color="C3", s=90, zorder=3, label="grid-worst")
ax.set_xlabel(r"$\log_{10}\lambda$")
ax.set_ylabel(r"Oracle RMSE$(\hat c - c^{\mathrm{true}})\,/\,\mathrm{sd}(c^{\mathrm{true}})$")
ax.set_title("HP cycle recovery vs smoothing parameter")
ax.grid(True, alpha=0.35)
ax.legend()
fig.tight_layout()
plt.show()


In [ ]:
def plot_overlay(ax, chat: np.ndarray, title: str) -> None:
    ax.plot(t, y, color="0.55", lw=1.0, alpha=0.9, label=r"$y$")
    ax.plot(t, c, color="0.15", lw=2.0, label=r"$c^{\mathrm{true}}$")
    ax.plot(t, chat, color="C0", lw=1.4, ls="--", label=r"$\hat c(\lambda)$")
    ax.set_xlabel("t")
    ax.set_ylabel("level")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right", fontsize=8)


fig, axes = plt.subplots(2, 1, figsize=(8.5, 6.5), sharex=True)
plot_overlay(
    axes[0],
    cycles[i_worst],
    f"Worst λ on grid: {lam_worst:.3g}  (log10={log10_lam[i_worst]:.3f})",
)
plot_overlay(
    axes[1],
    cycles[i_best],
    f"Best λ on grid: {lam_best:.3g}  (log10={log10_lam[i_best]:.3f})",
)
fig.tight_layout()
plt.show()


## Intuition: $\lambda$ too small vs too large

**Counts / shapes inspected:** sample length $T=240$; cycle $\hat{c}$ array shape `(T,)`; $\lambda$ grid length $10$ with `np.logspace(2, 9, num=10)` (roughly $10^2$ to $10^9$). Oracle curve and two overlay panels compare $y$, $c^{\mathrm{true}}$, and $\hat{c}(\lambda)$ at the grid-worst and grid-best $\lambda$.

**$\lambda$ too small:** the penalty on second differences of the fitted trend is weak, so $\hat{g}$ can hug the short-run wiggles in $y$. The estimated cycle $\hat{c}=y-\hat{g}$ then behaves like a high-pass residual: it still contains noise and fast variation that truly belonged in $\varepsilon$ (and can even trace the smooth trend), so $\hat{c}$ is a poor match to the structured oscillatory $c^{\mathrm{true}}$.

**$\lambda$ too large:** $\hat{g}$ is forced to be very smooth; low-frequency variation in $y$ that actually comes from $c^{\mathrm{true}}$ is absorbed into $\hat{g}$. Then $\hat{c}$ is overdamped—amplitude shrinks and phase/energy are wrong—so oracle error vs $c^{\mathrm{true}}$ grows again.

The grid-best $\lambda$ sits between these extremes for this particular DGP and noise level; it is **not** guaranteed to coincide with common quarterly macro defaults—that is a separate design choice.
